# Task 2 Demo - Multi-Method Clustering

In this notebook I demonstrate the clustering part of the assignment using multiple methods.

What I show:

1. Build clustering features from normalized text.
2. Run at least two clustering methods (I run three: K-Means, Agglomerative, Spectral).
3. Compare methods across different cluster counts (max 10).
4. Evaluate with silhouette score and export results.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans, SpectralClustering
from sklearn.metrics import silhouette_score

from data_mining_assignment.core.data_io import ArticleDataset
from data_mining_assignment.tasks.preprocessing import TextNormalizer, TextPreprocessor

## Load data and build clustering features

In [2]:
project_root_path = Path.cwd().parent
results_data_directory_path = project_root_path / "data" / "results"
results_data_directory_path.mkdir(parents=True, exist_ok=True)

articles_csv_path = project_root_path / "data" / "raw" / "articles.csv"
articles_data_frame = ArticleDataset(input_csv_path=articles_csv_path).load_articles()

text_normalizer = TextNormalizer()
normalized_bundle = text_normalizer.normalize_for_both_tasks(articles_data_frame["text"].tolist())

# Word-level TF-IDF for interpretable clustering.
clustering_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf",
    max_features=20000,
    min_document_frequency=1,
    max_document_frequency=0.92,
    ngram_range=(1, 2),
    analyzer_mode="word",
)

clustering_feature_matrix = clustering_preprocessor.fit_transform(normalized_bundle.clustering_texts)

# Dense LSA view for methods that work better in dense space.
lsa_preprocessor = TextPreprocessor(
    vectorization_model_name="tfidf_lsa_dense",
    max_features=20000,
    min_document_frequency=1,
    max_document_frequency=0.92,
    ngram_range=(1, 2),
    analyzer_mode="word",
    dense_embedding_dimension=128,
    random_seed=42,
)
lsa_feature_matrix = lsa_preprocessor.fit_transform(normalized_bundle.clustering_texts)

clustering_feature_matrix.shape, lsa_feature_matrix.shape

((2164, 20000), (2164, 128))

## Compare methods and cluster counts

In [3]:
candidate_cluster_counts = [4, 5, 6, 7, 8, 9, 10]

scored_rows: list[dict[str, float | int | str]] = []
labels_by_method_and_k: dict[tuple[str, int], np.ndarray] = {}

for cluster_count in candidate_cluster_counts:
    kmeans_model = KMeans(n_clusters=cluster_count, n_init="auto", random_state=42)
    kmeans_labels = kmeans_model.fit_predict(clustering_feature_matrix)
    kmeans_silhouette = silhouette_score(clustering_feature_matrix, kmeans_labels)
    scored_rows.append(
        {
            "method": "kmeans",
            "cluster_count": cluster_count,
            "silhouette_score": float(kmeans_silhouette),
        }
    )
    labels_by_method_and_k[("kmeans", cluster_count)] = kmeans_labels

    agglomerative_model = AgglomerativeClustering(n_clusters=cluster_count, metric="cosine", linkage="average")
    agglomerative_labels = agglomerative_model.fit_predict(lsa_feature_matrix)
    agglomerative_silhouette = silhouette_score(lsa_feature_matrix, agglomerative_labels)
    scored_rows.append(
        {
            "method": "agglomerative",
            "cluster_count": cluster_count,
            "silhouette_score": float(agglomerative_silhouette),
        }
    )
    labels_by_method_and_k[("agglomerative", cluster_count)] = agglomerative_labels

    try:
        spectral_model = SpectralClustering(
            n_clusters=cluster_count,
            affinity="nearest_neighbors",
            n_neighbors=12,
            assign_labels="kmeans",
            random_state=42,
        )
        spectral_labels = spectral_model.fit_predict(lsa_feature_matrix)
        spectral_silhouette = silhouette_score(lsa_feature_matrix, spectral_labels)
        scored_rows.append(
            {
                "method": "spectral",
                "cluster_count": cluster_count,
                "silhouette_score": float(spectral_silhouette),
            }
        )
        labels_by_method_and_k[("spectral", cluster_count)] = spectral_labels
    except Exception as spectral_error:
        scored_rows.append(
            {
                "method": "spectral",
                "cluster_count": cluster_count,
                "silhouette_score": np.nan,
                "error": str(spectral_error),
            }
        )

clustering_metrics_data_frame = pd.DataFrame(scored_rows).sort_values(
    ["method", "silhouette_score"], ascending=[True, False]
)
clustering_metrics_data_frame.to_csv(results_data_directory_path / "notebook_05_clustering_method_metrics.csv", index=False)
clustering_metrics_data_frame.head(15)

,method,cluster_count,silhouette_score
4,agglomerative,5,0.243182
1,agglomerative,4,0.240553
10,agglomerative,7,0.234552
7,agglomerative,6,0.231768
13,agglomerative,8,0.081431
16,agglomerative,9,0.052500
19,agglomerative,10,0.050589
12,kmeans,8,0.012176
18,kmeans,10,0.011949
15,kmeans,9,0.011400


## Save best label set per method

In [4]:
best_rows = (
    clustering_metrics_data_frame.dropna(subset=["silhouette_score"])
    .sort_values(["method", "silhouette_score"], ascending=[True, False])
    .groupby("method", as_index=False)
    .first()
)

best_label_table = pd.DataFrame({"doc_id": articles_data_frame["doc_id"].tolist()})

for _, row in best_rows.iterrows():
    method_name = str(row["method"])
    selected_cluster_count = int(row["cluster_count"])
    selected_labels = labels_by_method_and_k[(method_name, selected_cluster_count)]
    best_label_table[f"{method_name}_k{selected_cluster_count}"] = selected_labels

best_label_table.to_csv(results_data_directory_path / "notebook_05_best_method_labels.csv", index=False)
best_rows.to_csv(results_data_directory_path / "notebook_05_best_method_summary.csv", index=False)

best_rows

,method,cluster_count,silhouette_score
0,agglomerative,5,0.243182
1,kmeans,8,0.012176
2,spectral,5,0.246710


## Cluster size diagnostics for each best method

In [5]:
cluster_size_rows: list[dict[str, int | str]] = []

for _, row in best_rows.iterrows():
    method_name = str(row["method"])
    selected_cluster_count = int(row["cluster_count"])
    selected_labels = labels_by_method_and_k[(method_name, selected_cluster_count)]
    label_counts = pd.Series(selected_labels).value_counts().sort_index()
    for label_value, label_count in label_counts.items():
        cluster_size_rows.append(
            {
                "method": method_name,
                "cluster_count": selected_cluster_count,
                "label": int(label_value),
                "document_count": int(label_count),
            }
        )

cluster_size_table = pd.DataFrame(cluster_size_rows)
cluster_size_table.to_csv(results_data_directory_path / "notebook_05_cluster_size_table.csv", index=False)
cluster_size_table.head(20)


,method,cluster_count,label,document_count
0,agglomerative,5,0,2099
1,agglomerative,5,1,50
2,agglomerative,5,2,8
3,agglomerative,5,3,3
4,agglomerative,5,4,4
5,kmeans,8,0,70
6,kmeans,8,1,880
7,kmeans,8,2,112
8,kmeans,8,3,298
9,kmeans,8,4,192
